# 360 City Analyser on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JSempereH/360_city_analyser/blob/main/notebooks/colab_quickstart.ipynb)

Small-scale validation of the whole chain on a GPU: download panoramas, let the survey agent walk the area tile by tile,
analyze every panorama, describe each building with a vision-language model (GEM v3 structural material and lateral
load-resisting system, storeys, use, facade, age), then open the viewer here and click a footprint or a building.

Runtime > Change runtime type > **T4 GPU** (or better). Run the cells in order. Every command stops the notebook with
its own error message when it fails, so the first red cell is the one to read.

## 1. Drive, repository and environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/city-analyser'      # datasets and caches persist here
os.makedirs(WORK + '/data', exist_ok=True)

In [ ]:
import shlex, subprocess

def sh(command, cwd='/content/repo'):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print('$', command, flush=True)
    process = subprocess.Popen(command, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    if process.wait() != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}: {command}')

def secret(name):
    """A Colab secret (key icon on the left; enable notebook access), or None."""
    try:
        from google.colab import userdata
        return (userdata.get(name) or '').strip().strip('"').strip("'") or None  # pasted values often carry spaces or quotes
    except Exception:
        return None

In [ ]:
# A public repository needs no token. For a private fork, store a read-only token in Colab secrets as GITHUB_TOKEN.
REPOSITORY = 'github.com/JSempereH/360_city_analyser.git'
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPOSITORY}' if token else f'https://{REPOSITORY}'
sh('rm -rf /content/repo', cwd='/content')
sh(f'git clone --depth 1 {url} /content/repo', cwd='/content')

In [ ]:
sh('pip -q install uv')
sh('uv sync --extra cu126')
# data/ lives on Drive. Model weights stay on the session disk: Drive does not keep the Hugging Face cache's
# symlinks and can serve a file still being synced, which breaks loading (SafetensorError: header too large).
sh(f'rm -rf data models && ln -s {WORK}/data data && mkdir -p /content/models && ln -s /content/models models')
sh('uv run python -c "import torch; assert torch.cuda.is_available(), \'no GPU: Runtime > Change runtime type\'; print(torch.cuda.get_device_name(0))"')

In [ ]:
os.environ['BUILDING_ANALYSIS_MODEL'] = 'high'                        # for a 16 GB T4; use 'balanced' if memory runs out
os.environ['BUILDING_ANALYSIS_STAGE_CACHE_DIR'] = WORK + '/stage-cache'
os.environ['BUILDING_ANALYSIS_OSM_CACHE_DIR'] = WORK + '/osm-cache'    # Colab can reach Overpass; the cache avoids repeating requests
sh('uv run city-analyser-models')

## 2. Data

Option A: copy a dataset folder from your laptop (for example `data/valencia-centro`, with its `manifest.json` and
`images/`) to `MyDrive/city-analyser/data/` and skip the download cell.

Option B: download it from Mapillary. Store your token in Colab secrets as `MAPILLARY_ACCESS_TOKEN` and enable notebook
access for it. For a first validation keep the area and `MAX_IMAGES` small; rerunning the cell skips files already downloaded.

In [ ]:
DATASET = 'valencia-centro'            # folder name under MyDrive/city-analyser/data
BBOX = '-0.3780,39.4720,-0.3740,39.4750'   # min_lon,min_lat,max_lon,max_lat: a few streets of Valencia's old town
MAX_IMAGES = 40

In [ ]:
mapillary_token = secret('MAPILLARY_ACCESS_TOKEN')
if not mapillary_token:
    raise RuntimeError('No MAPILLARY_ACCESS_TOKEN secret: add it under the key icon and enable notebook access, or use option A.')
import urllib.error, urllib.parse, urllib.request
try:  # one tiny query first, so a wrong token fails here with a clear message
    urllib.request.urlopen('https://graph.mapillary.com/images?' + urllib.parse.urlencode({'access_token': mapillary_token, 'bbox': BBOX, 'fields': 'id', 'limit': 1}), timeout=30)
except urllib.error.HTTPError as error:
    if error.code in (400, 401, 403):
        raise RuntimeError(f'Mapillary rejected the MAPILLARY_ACCESS_TOKEN secret (HTTP {error.code}). It must be the client access token, '
                           'MLY|<app id>|<32 hex characters>, from mapillary.com/dashboard/developers, not the client secret.') from None
    raise
os.environ['MAPILLARY_ACCESS_TOKEN'] = mapillary_token
sh(f'uv run city-analyser-download --tiled-bbox={BBOX} --output data/{DATASET} --max-images {MAX_IMAGES} --image-size 2048 --workers 8')

In [ ]:
import json
from pathlib import Path
manifest = Path(f'/content/repo/data/{DATASET}/manifest.json')
if not manifest.is_file():
    raise RuntimeError(f'{manifest} does not exist: the download did not run or failed (see its output above), or the copied folder has another name.')
images = [image for image in json.loads(manifest.read_text())['images'] if image.get('local_file')]
print(f'{DATASET}: {len(images)} downloaded panoramas')
if not images:
    raise RuntimeError('The dataset has no downloaded panoramas: try a larger area or another place with Mapillary 360 coverage.')

## 3. Survey: the agent walks the area and describes every building

`city-analyser-survey` splits the panoramas into square tiles and visits them in a serpentine order. In each tile it
analyzes the panoramas (segmentation, OSM footprints, association), then, for every building whose views are all
analyzed, exports its best crops, writes the OSM-tag prior and asks the vision-language model, which first writes
what it observes and then answers with GEM v3 codes. Results go to `data/<dataset>/attributes/`.

The model loads in this process with Transformers. `Qwen/Qwen3-VL-4B-Instruct` (Apache-2.0, about 9 GB in fp16) fits a
T4 next to the analysis models; use `Qwen/Qwen3-VL-2B-Instruct` if memory runs out, or `Qwen/Qwen3-VL-8B-Instruct` on a
24 GB GPU. Weights go to the session disk, not to Drive.

`--max-tiles` and `--max-buildings` keep this first run small. Running the cell again continues where it stopped:
analyses, crops and model answers that are still current are reused.

In [ ]:
VLM_MODEL = 'Qwen/Qwen3-VL-4B-Instruct'
sh(f'uv run city-analyser-survey {DATASET} --vlm-model {VLM_MODEL} --vlm-cache-dir /content/vlm-cache '
   '--tile-size 150 --max-tiles 2 --max-buildings 15 --skip-low-quality')

### Check the answers

One line per described building: the GEM string, where each value comes from and the model's own observations.
Look at the crops: an answer is only as good as the view it saw.

In [ ]:
import sys
sys.path.insert(0, '/content/repo/src')
from urllib.parse import unquote
from IPython.display import HTML, display
from city_analyser.attributes.card import building_card

dataset_dir = Path(f'/content/repo/data/{DATASET}')
state = json.loads((dataset_dir / 'survey-state.json').read_text())
print(state['summary'])
rows = []
for building_id in sorted(state['buildings'])[:15]:
    card = building_card(dataset_dir, DATASET, building_id)
    if card is None:
        continue
    values = {name: ('-' if entry.get('abstained') else f"{entry.get('value')} ({'OSM' if entry.get('source') == 'osm_tag' else 'AI'})")
              for name, entry in card['attributes'].items()}
    observation = card['observations'][0] if card['observations'] else {}
    crop = dataset_dir / unquote(observation['crop_url']).split(f'/{DATASET}/', 1)[1] if observation.get('crop_url') else None
    image = f'<img src="data:image/jpeg;base64,{__import__("base64").b64encode(crop.read_bytes()).decode()}" width="120">' if crop and crop.is_file() else ''
    rows.append(f"<tr><td>{image}</td><td><b>{building_id}</b><br><code>{card['gem']['string']}</code></td>"
                f"<td>{'<br>'.join(f'{k}: {v}' for k, v in values.items())}</td><td>{observation.get('text', '')}</td></tr>")
display(HTML('<table>' + ''.join(rows) + '</table>'))

## 4. Open the viewer here

The viewer runs in the background and opens in a new browser tab through Colab's proxy. Click a footprint on the map or
a building in the panorama: the card shows the GEM string, each attribute with its source, and the model's observations
under "Why the model thinks so".

The **Assistant** tab talks with the model about the selected building (it sees the photos and the record) and
**Analyze now** has it describe the building again while you watch. Layers > Building colors paints the footprints by GEM material, lateral system, floors or review status. The card's
**Review** button records your own decision, which wins over the model; Review > Show > Structure to check lists the
answers worth a look, weakest first. Enter your name in the Review tab first: it is stored with each review.

In [ ]:
viewer_log = open('/content/viewer.log', 'w')
# The assistant (Assistant tab, Analyze now, live presentation) uses the same model, loaded by the viewer.
viewer_env = {**os.environ, 'ATTRIBUTES_VLM_BACKEND': 'transformers', 'ATTRIBUTES_VLM_MODEL': VLM_MODEL, 'ATTRIBUTES_VLM_CACHE_DIR': '/content/vlm-cache'}
viewer = subprocess.Popen(['uv', 'run', 'city-analyser-viewer', '--host', '127.0.0.1', '--port', '8765'],
                          cwd='/content/repo', stdout=viewer_log, stderr=subprocess.STDOUT, env=viewer_env)
import time, urllib.request
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8765/api/health', timeout=2); break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError('The viewer did not start; see /content/viewer.log')
from google.colab.output import eval_js
from IPython.display import HTML, display
proxy = eval_js('google.colab.kernel.proxyPort(8765)')   # https://<id>-8765-colab.googleusercontent.com/, for your Google account only
link = f"{proxy.rstrip('/')}/viewer/?dataset={DATASET}"
display(HTML(f'<a href="{link}" target="_blank" rel="noopener">Open the viewer in a new tab</a><br><code>{link}</code>'))

### Take the results to your laptop

The export packs the analyses, crops and model answers (and, with `--with-images`, the panoramas). On your laptop,
`uv run city-analyser-results import results-<dataset>.zip` unpacks it into `data/` without touching the reviews you
made there, and prints the settings to start the viewer with. There you can correct any value by hand (Edit values) or
ask a local model (Ollama, vLLM) about it.

In [ ]:
sh(f'uv run city-analyser-results export {DATASET} {WORK}/results-{DATASET}.zip --with-images')
from google.colab import files
files.download(f'{WORK}/results-{DATASET}.zip')   # or take it from Drive

## 5. Larger runs, exports and evaluation

Remove `--max-tiles` and `--max-buildings` to survey the whole dataset. For long runs start it in the background, so a
closed browser tab does not stop it, and follow the log; a lost session is resumed by running the same command again.

In [ ]:
log_path = f'{WORK}/survey-{DATASET}.log'
survey = subprocess.Popen(f'uv run city-analyser-survey {DATASET} --vlm-model {VLM_MODEL} --vlm-cache-dir /content/vlm-cache '
                          f'--tile-size 150 --skip-low-quality >> {log_path} 2>&1', shell=True, cwd='/content/repo')

In [ ]:
print('running' if survey.poll() is None else f'finished with code {survey.returncode} (0 = ok, 1 = aborted, 130 = interrupted)')
sh(f'tail -n 15 {log_path}')

In [ ]:
# Per-building table with the GEM columns (GeoJSON, CSV, GeoParquet) and agreement with OSM tags where they exist
sh('uv sync --extra cu126 --extra parquet')
provider = 'vlm_' + VLM_MODEL.replace('/', '_')
sh(f'uv run city-analyser-export {DATASET} --provider {provider}')
sh(f'uv run python -m evaluation.attributes --dataset {DATASET} --provider {provider} | head -60')

### Optional: a vLLM server instead of Transformers

For thousands of buildings a model server batches far better. vLLM goes in its own environment (do not mix its torch
with the project's); the survey then calls it on `localhost`. It shares the GPU with the analysis models, so use a GPU
with 24 GB or more (L4, A100); on a T4 stay with Transformers. Use it instead of the background survey above, not
next to it: two surveys on one dataset at once would repeat each other's work.

In [ ]:
gpu_gib = int(subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
                             capture_output=True, text=True, check=True).stdout.split()[0]) / 1024
if gpu_gib < 20:
    raise RuntimeError(f'This GPU has {gpu_gib:.0f} GB: too little for vLLM next to the analysis models. '
                       'Use the Transformers survey above, or pick an L4 or A100 runtime.')
if 'survey' in globals() and survey.poll() is None:
    survey.terminate(); survey.wait()   # one survey per dataset at a time; this one resumes where that one stopped
sh('uv venv /content/vllm-env && uv pip install --python /content/vllm-env/bin/python vllm', cwd='/content')
vllm = subprocess.Popen(['/content/vllm-env/bin/vllm', 'serve', VLM_MODEL, '--dtype', 'half', '--max-model-len', '8192',
                         '--gpu-memory-utilization', '0.6', '--port', '8000'],   # leaves room for the analysis models
                        stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)

In [ ]:
import time, urllib.request
for _ in range(180):   # loading the weights takes a few minutes the first time
    if vllm.poll() is not None:
        sh("grep -n -i -E 'error|out of memory|cache blocks' /content/vllm.log | grep -v -i warning | head -15 || true")   # the cause, not the end of the traceback
        raise RuntimeError('vLLM exited; the lines above say why (full log: /content/vllm.log)')
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2); print('vLLM is ready'); break
    except OSError:
        time.sleep(5)
else:
    sh("grep -n -i -E 'error|out of memory|cache blocks' /content/vllm.log | grep -v -i warning | head -15 || true")
    raise RuntimeError('vLLM did not start in 15 minutes (full log: /content/vllm.log)')

In [ ]:
sh(f'uv run city-analyser-survey {DATASET} --vlm-backend openai --vlm-url http://127.0.0.1:8000 --vlm-model {VLM_MODEL} --tile-size 150')

## 6. Results on your laptop

Everything stays on Drive under `city-analyser/data/<dataset>` (`analysis`, `crops`, `attributes`, `exports`,
`survey-state.json`). Copy that folder to `data/` in your checkout and run `make viewer`.